# MIR — ResNet50 (Zero-Shot Uniquement)

Ce notebook évalue les performances d'un modèle **ResNet50** pré-entraîné sur ImageNet pour une tâche de recherche d'images (Retrieval), sans aucun finetuning.

Méthodologie :
1. Chargement de ResNet50 en tant qu'extracteur de caractéristiques (Features Extractor).
2. Extraction et normalisation des embeddings.
3. Évaluation zero-shot (Recall@K + mAP).
4. Sauvegarde des embeddings, du graphique de recall et des résultats JSON sur Google Drive.

Toutes les sauvegardes vont dans le dossier `ResNet50_ZeroShot_Evaluation` sur le Drive.

In [ ]:
# ── Configuration des chemins
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
IMAGE_DIR    = os.path.join(PROJECT_ROOT, "dataset")
SAVE_DIR     = os.path.join(PROJECT_ROOT, "results", "ResNet50_training")
os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 64

print(f'Dataset : {IMAGE_DIR}')
print(f'Sauvegardes : {SAVE_DIR}')
print(f'GPU : {os.popen("nvidia-smi --query-gpu=name --format=csv,noheader").read().strip()}')

In [ ]:
# ── Imports communs et fonctions helper
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Utilisation de : {device}')

# Extraction du label depuis le nom de fichier
def get_label_from_filename(filename):
    name_without_ext = filename.split('.')[0]
    parts = name_without_ext.split('_')
    if len(parts) >= 4:
        return f'{parts[2]}_{parts[3]}'
    return 'Label_Inconnu'

In [ ]:
# ── Charger ResNet50 pré-entraîné comme extracteur de features
class ResNet50FeatureExtractor(nn.Module):
    """ResNet50 sans la tête de classification. Sortie : 2048-dim."""
    def __init__(self):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT
        base = models.resnet50(weights=weights)

        # On garde tout sauf la dernière couche (fc)
        self.features = nn.Sequential(*list(base.children())[:-1])

    def forward(self, x):
        x = self.features(x)
        return torch.flatten(x, 1)  # (B, 2048)

model_zs = ResNet50FeatureExtractor().to(device)
model_zs.eval()

n_params = sum(p.numel() for p in model_zs.parameters())
print(f'ResNet50 chargé. Paramètres : {n_params/1e6:.2f}M')
print('Dimension de sortie : 2048')

In [ ]:
# ── Dataset et DataLoader pour l'extraction
class SimpleImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_filenames = [
            f for f in os.listdir(root_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))
        ]

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_filenames[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_filenames[idx]

# Transformations standard ImageNet
transform_eval = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

dataset    = SimpleImageDataset(root_dir=IMAGE_DIR, transform=transform_eval)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4)

print(f'Dataset chargé : {len(dataset)} images')

In [ ]:
# ── Extraction des embeddings zero-shot
all_embeddings = []
all_filenames  = []

print(f'Extraction des vecteurs pour {len(dataset)} images...')

with torch.no_grad():
    for images, filenames in tqdm(dataloader):
        images = images.to(device)
        embeddings = model_zs(images)
        embeddings = F.normalize(embeddings, p=2, dim=1)  
        all_embeddings.append(embeddings.cpu())
        all_filenames.extend(filenames)

database_tensor = torch.cat(all_embeddings, dim=0)

save_path = os.path.join(SAVE_DIR, 'ResNet50_zero_shot.pth')
torch.save({'embeddings': database_tensor, 'filenames': all_filenames}, save_path)

print(f'Base de données créée : {database_tensor.shape}')
print(f'Sauvegardée : {save_path}')

In [ ]:
# ── Évaluation Recall@K + mAP zero-shot et affichage
db         = torch.load(os.path.join(SAVE_DIR, 'ResNet50_zero_shot.pth'))
embeddings = db['embeddings']
filenames  = db['filenames']
labels     = [get_label_from_filename(f) for f in filenames]

K_values    = [1, 5, 20, 50]
recalls     = {k: 0 for k in K_values}
ap_scores   = []
num_queries = len(embeddings)

print(f'Évaluation du Recall sur {num_queries} images...')

similarity_matrix = torch.mm(embeddings, embeddings.T)

for i in range(num_queries):
    query_label = labels[i]
    scores      = similarity_matrix[i].clone()
    scores[i]   = -1.0 

    _, top_indices   = torch.topk(scores, max(K_values))
    retrieved_labels = [labels[idx.item()] for idx in top_indices]

    
    for k in K_values:
        if query_label in retrieved_labels[:k]:
            recalls[k] += 1

    
    num_relevant = labels.count(query_label) - 1
    if num_relevant == 0:
        continue

    hits, precision_sum = 0, 0.0
    for rank, lbl in enumerate(retrieved_labels, start=1):
        if lbl == query_label:
            hits += 1
            precision_sum += hits / rank
    ap_scores.append(precision_sum / num_relevant)

recall_scores = {k: recalls[k] / num_queries * 100 for k in K_values}
mAP           = sum(ap_scores) / len(ap_scores) * 100

print('\n--- Résultats Zero-Shot ResNet50 ---')
for k, v in recall_scores.items():
    print(f'Recall@{k:>2} : {v:.2f}%')
print(f'mAP        : {mAP:.2f}%')

# ── Graphique
plt.figure(figsize=(10, 6))
plt.plot(K_values, list(recall_scores.values()),
         marker='o', linestyle='-', color='#2ca02c',
         linewidth=2.5, markersize=8, label='ResNet50 (Zero-Shot)')

for k, v in recall_scores.items():
    plt.annotate(f'{v:.1f}%', (k, v),
                 textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=10)

plt.text(0.98, 0.05, f'mAP : {mAP:.2f}%',
         transform=plt.gca().transAxes, fontsize=11,
         ha='right', va='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.title('Courbe de Rappel (Recall@K) - ResNet50 Zero-Shot',
          fontsize=14, fontweight='bold')
plt.xlabel('Rang K (Nombre de résultats affichés)', fontsize=12)
plt.ylabel('Taux de Rappel (%)', fontsize=12)
plt.xticks(K_values)
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()

# ── Sauvegardes
graph_filename = 'recall_curve_ResNet50_zero_shot'
graph_path     = os.path.join(SAVE_DIR, f'{graph_filename}.png')
result_path    = os.path.join(SAVE_DIR, f'results_{graph_filename}.json')

plt.savefig(graph_path, dpi=300, bbox_inches='tight')

results = {
    'model':    graph_filename,
    'recall':   {f'@{k}': round(v, 2) for k, v in recall_scores.items()},
    'mAP':      round(mAP, 2),
    'n_images': num_queries,
}

with open(result_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nGraphique sauvegardé : {graph_path}')
print(f'Résultats sauvegardés : {result_path}')
plt.show()